In [1]:
import subprocess
import os

# 1. Install python3.10 and venv support on Colab
subprocess.run(["sudo", "apt-get", "update", "-y"], check=True)
subprocess.run(["sudo", "apt-get", "install", "python3.10", "python3.10-venv", "python3.10-dev", "-y"], check=True)

# 2. Create the virtual environment
subprocess.run(["python3.10", "-m", "venv", "/content/venv"], check=True)

# 3. Upgrade pip inside the venv
subprocess.run(["/content/venv/bin/python", "-m", "pip", "install", "--upgrade", "pip"], check=True)

# 4. Install the serving pins + autoawq for Day 4
subprocess.run([
    "/content/venv/bin/python", "-m", "pip", "install", "-q",
    "vllm==0.6.*",
    "transformers==4.46.*",
    "accelerate==1.1.*",
    "autoawq==0.2.*",
    "httpx==0.27.*",
    "openai==1.54.*"
], check=True)

print("Python 3.10 venv ready with AWQ serving pins installed!")

Python 3.10 venv ready with AWQ serving pins installed!


In [3]:
# 1. Install pins and serve AWQ
import subprocess, sys

VLLM_PIN = "0.6.*"
BITSANDBYTES_PIN = "0.49.2"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    # Use the python executable from the venv created in the previous cell
    cmd = ["/content/venv/bin/python", "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

pip_install(
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"autoawq=={AUTOAWQ_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
)
print("serving pins installed")

# 2. Launch server cell for AWQ
import os, signal, subprocess, time

MODEL = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

def build_cmd(args: dict) -> list:
    # Use the python executable from the venv created in the previous cell
    cmd = ["/content/venv/bin/python", "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    proc = subprocess.Popen(
        cmd, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True,
    )
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server(SERVER_ARGS)

# 3. Health poll cell
import urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            return "".join(fh.readlines()[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    print("server did not come up. common causes: model still downloading or OOM.")
    return False

healthy = wait_for_health()

installing: vllm==0.6.* transformers==4.46.* accelerate==1.1.* autoawq==0.2.* httpx==0.27.* openai==1.54.*
serving pins installed
launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 9465, logging to /content/server.log
server healthy after about 78s: http://localhost:8000/v1/models -> 200


In [4]:
# Cell 2: measure AWQ VRAM and tokens/s
import time
import subprocess
from openai import OpenAI

# 1. Measure VRAM
print("--- GPU Memory Used ---")
subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"])

# 2. Measure Tokens/s
print("\n--- Speed Test ---")
client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")

start_time = time.time()
response = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    messages=[{"role": "user", "content": "Write a long, detailed story about a time traveler."}],
    max_tokens=200
)
end_time = time.time()

tokens = response.usage.completion_tokens
time_taken = end_time - start_time
tokens_per_s = tokens / time_taken

print(f"Generated {tokens} tokens in {time_taken:.2f} seconds.")
print(f"Speed: {tokens_per_s:.2f} tokens/s")

--- GPU Memory Used ---

--- Speed Test ---
Generated 200 tokens in 3.15 seconds.
Speed: 63.45 tokens/s


In [5]:
# Cell 3: five-prompt quality spot check
from openai import OpenAI

SPOT_PROMPTS = [
    "Write a two-sentence summary of what an inference server does.",
    "A user asks for the weather in Riyadh and the time in Tokyo. "
    "What two tool calls would you make?",
    "Refactor this into a single sentence: The GPU was busy but not "
    "productive, because decode is memory-bound.",
    "List the steps to roll back a bad deployment, in order.",
    "Explain quantisation to a non-technical manager in three sentences.",
]

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")

for p in SPOT_PROMPTS:
    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct-AWQ",
        messages=[{"role": "user", "content": p}], max_tokens=200)
    print("PROMPT:", p[:50], "...")
    print(r.choices[0].message.content, "\n")
    print("-" * 40)

PROMPT: Write a two-sentence summary of what an inference  ...
An inference server is a software that processes and responds to model predictions, enabling real-time applications, machine learning, and AI-driven businesses. It efficiently takes model outputs and translates them into specific actions or responses required by the application. 

----------------------------------------
PROMPT: A user asks for the weather in Riyadh and the time ...
To provide weather information for Riyadh and the time in Tokyo, you would need to use an external weather API for Riyadh and an external time API for Tokyo. The specific API calls would look like this:

For the weather in Riyadh:
```python
response = get_weather_api('Riyadh')
print(response)
```

For the time in Tokyo:
```python
response = get_time_api('Tokyo')
print(response)
``` 

----------------------------------------
PROMPT: Refactor this into a single sentence: The GPU was  ...
The GPU was working hard but not efficient due to decode ope

In [6]:
# Cell 3: five-prompt quality spot check
from openai import OpenAI

SPOT_PROMPTS = [
    "Write a two-sentence summary of what an inference server does.",
    "A user asks for the weather in Riyadh and the time in Tokyo. "
    "What two tool calls would you make?",
    "Refactor this into a single sentence: The GPU was busy but not "
    "productive, because decode is memory-bound.",
    "List the steps to roll back a bad deployment, in order.",
    "Explain quantisation to a non-technical manager in three sentences.",
]

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")

for p in SPOT_PROMPTS:
    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct-AWQ",
        messages=[{"role": "user", "content": p}], max_tokens=200)
    print("PROMPT:", p[:50], "...")
    print(r.choices[0].message.content, "\n")
    print("-" * 40)

PROMPT: Write a two-sentence summary of what an inference  ...
An inference server takes in requests from client devices and runs machine learning models to return a prediction or inference. 

----------------------------------------
PROMPT: A user asks for the weather in Riyadh and the time ...
To obtain the weather information in Riyadh, I would make a call to the weather tool with the query "weather in Riyadh". After receiving the response with the current weather conditions, temperature, humidity, and other relevant details, I can share this information with the user.

To obtain the time in Tokyo, I would make a call to the time tool with the query "time in Tokyo". The response will include not only the current time but also the local time zone offset. The user can then combine this information with the weather details to have a more comprehensive understanding of their surroundings in both locations.

To use these tools, the commands would be structured as follows:

Windows/Q/A:
`

In [7]:
from openai import OpenAI

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate an arithmetic expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string",
                                   "description": "e.g. 23 * 19"},
                },
                "required": ["expression"],
            },
        },
    },
]

CANONICAL = [
    {
        "id": "two_tool",
        "k": 4,
        "wants_call": True,
        "prompt": "What is the weather in Riyadh, and what is 23 multiplied "
                  "by 19? Use your tools.",
    },
    {
        "id": "single",
        "k": 4,
        "wants_call": True,
        "prompt": "What is the weather in Tokyo right now? Use your tools.",
    },
    {
        "id": "distractor",
        "k": 2,
        "wants_call": False,
        "prompt": "In one sentence, explain what a tool call is. Do not call "
                  "any tool; just answer.",
    },
]

def _tool_calls_of(message) -> list:
    tc = getattr(message, "tool_calls", None)
    return list(tc) if tc else []

def _valid_call(call) -> bool:
    import json
    try:
        fn = call.function.name
        if fn not in ("get_weather", "calculate"):
            return False
        args = json.loads(call.function.arguments or "{}")
    except (AttributeError, ValueError):
        return False
    if fn == "get_weather":
        return isinstance(args.get("city"), str) and bool(args["city"])
    if fn == "calculate":
        return isinstance(args.get("expression"), str) and bool(args["expression"])
    return False

def run_smoke(base_url: str, model: str, temperature: float = 0.0) -> dict:
    client = OpenAI(base_url=base_url, api_key="not-needed")
    total_attempts = 0
    valid_call_attempts = 0
    distractor_attempts = 0
    distractor_call_free = 0
    per_prompt = {}

    for spec in CANONICAL:
        pid, k, wants = spec["id"], spec["k"], spec["wants_call"]
        got_valid = 0
        got_call_free = 0
        for _ in range(k):
            total_attempts += 1
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": spec["prompt"]}],
                tools=TOOLS,
                tool_choice="auto",
                temperature=temperature,
                max_tokens=256,
            )
            msg = resp.choices[0].message
            calls = _tool_calls_of(msg)
            any_valid = any(_valid_call(c) for c in calls)

            if wants:
                if any_valid:
                    valid_call_attempts += 1
                    got_valid += 1
            else:
                distractor_attempts += 1
                if not calls:
                    valid_call_attempts += 1
                    distractor_call_free += 1
                    got_call_free += 1

        per_prompt[pid] = {"k": k, "wants_call": wants,
                           "valid": got_valid, "call_free": got_call_free}

    distractor_majority = (distractor_call_free * 2 > distractor_attempts) \
        if distractor_attempts else True
    passed = (valid_call_attempts >= 8) and distractor_majority

    return {
        "model": model,
        "total_attempts": total_attempts,
        "score": valid_call_attempts,
        "distractor_attempts": distractor_attempts,
        "distractor_call_free": distractor_call_free,
        "distractor_majority_clean": distractor_majority,
        "per_prompt": per_prompt,
        "passed": passed,
    }

# Run the test against our AWQ model
result = run_smoke(base_url="http://localhost:8000/v1",
                   model="Qwen/Qwen2.5-1.5B-Instruct-AWQ")
print(result)

{'model': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ', 'total_attempts': 10, 'score': 10, 'distractor_attempts': 2, 'distractor_call_free': 2, 'distractor_majority_clean': True, 'per_prompt': {'two_tool': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'single': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'distractor': {'k': 2, 'wants_call': False, 'valid': 0, 'call_free': 2}}, 'passed': True}


In [8]:
import json

with open("smoke_result.json", "w") as f:
    json.dump(result, f, indent=2)

print(" smoke_result.json")

 smoke_result.json


In [10]:
# Cell 6: clean shutdown
import os, signal

def shutdown_server(proc=None, port=8000):
    try:
        proc = server if proc is None else proc
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        print(f"sent SIGTERM to process group of pid {proc.pid}")
    except (ProcessLookupError, NameError):
        print("no server process to kill")

shutdown_server()

sent SIGTERM to process group of pid 9465


In [16]:
content = """Model id: Qwen/Qwen2.5-1.5B-Instruct-AWQ
Quantisation: AWQ
Flags: --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
Smoke score: 10"""

with open("model-lock.md", "w") as f:
    f.write(content)

In [17]:
# Green-check verifier for Lab W3D4 (quantise and lock).
# Paste this as the last cell of your day-4 notebook and run it. It reads
# smoke_result.json (written from the smoke test) and model-lock.md, and checks
# that the smoke score meets the gate and that the lock file is fully filled in.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os, re


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    # 1) smoke result
    if not os.path.exists("smoke_result.json"):
        fail("smoke_result.json not found; write it in Cell 5")
    try:
        with open("smoke_result.json") as fh:
            result = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"smoke_result.json is not valid JSON: {exc}")

    for key in ("score", "total_attempts", "distractor_majority_clean", "passed"):
        if key not in result:
            fail(f"smoke_result.json missing key: {key}")

    score = result["score"]
    total = result["total_attempts"]
    if not isinstance(score, int) or not isinstance(total, int):
        fail("score and total_attempts must be integers")
    if total != 10:
        fail(f"total_attempts is {total}, the smoke test defines n=10")
    if score < 8:
        fail(f"smoke score {score}/10 is below the 8/10 gate")
    if not result["distractor_majority_clean"]:
        fail("distractor did not stay call-free in the majority; a model that "
             "always calls a tool fails the real consumer")
    if not result["passed"]:
        fail("smoke test reports passed=false")

    # 2) model-lock.md fully filled in
    if not os.path.exists("model-lock.md"):
        fail("model-lock.md not found")
    with open("model-lock.md") as fh:
        lock = fh.read()
    remaining = re.findall(r"FILL:", lock)
    if remaining:
        fail(f"model-lock.md has {len(remaining)} unfilled FILL: placeholders")
    # require a concrete model id line
    if not re.search(r"Model id:\s*\S+", lock):
        fail("model-lock.md has no concrete Model id")

    print(f"smoke score: {score}/{total}, distractor clean: "
          f"{result['distractor_majority_clean']}")
    print("model-lock.md: all fields filled")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


smoke score: 10/10, distractor clean: True
model-lock.md: all fields filled
GREEN CHECK: PASS
